# Colab Session: opportunity_2h_h48
Generated from colab-cli history log.

**Session Created**: 2026-10-12 06:52:34
- Endpoint: `m-s-kkb-usw1c1-19694otmrcu0q`
- Hardware: `NONE`

In [ ]:
# CELL: ENV
# Colab ships pandas/numpy; install tabular model extras.
%pip install -q --upgrade-strategy only-if-needed pyarrow requests scikit-learn xgboost lightgbm catboost

import json
import math
import pickle
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from xgboost import XGBClassifier, XGBRegressor

print("sklearn / xgboost / lightgbm / catboost ready")


   ━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/97.1 MB 108.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━ 41.4/97.1 MB 172.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━ 64.2/97.1 MB 97.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━ 94.2/97.1 MB 97.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 95.7 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 95.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 95.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 95.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


sklearn / xgboost / lightgbm / catboost ready


In [ ]:
# CELL: CONFIG
from pathlib import Path

RESOLUTION_MINUTES: Dict[str, int] = {
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
    "4h": 240,
    "6h": 360,
}

CONFIG: Dict[str, Any] = {
    "symbol": "BTCUSD",
    "base_url": "https://api.india.delta.exchange",
    "history_days": 900,
    "resolution": "2h",
    "horizon_bars": 48,
    "seed": 42,
    # Bracket. TP must clear the round-trip cost by a wide margin, otherwise no
    # classifier can be profitable (see the notebook header).
    "tp_atr_mult": 4.0,
    "sl_atr_mult": 2.0,
    "atr_period": 14,
    "round_trip_cost_pct": 0.0016,
    # Gate: median TP distance >= ratio x cost. Signal rule: a candle is only
    # eligible when its own TP distance >= ratio x cost (quiet-market guard).
    "min_tp_cost_ratio": 4.0,
    "minority_rate": 0.15,
    "ambiguous_win_max": 0.50,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    # Decision rule: trade only when P(side) clears a threshold chosen on
    # calibration rows. Candidates are probability quantiles of the calibration set.
    "calib_fraction": 0.15,
    "min_signals": 100,
    "signal_quantiles": [0.99, 0.98, 0.97, 0.95, 0.92, 0.90, 0.85, 0.80],
    "walk_forward_folds": 4,
    # Promotion gate thresholds (held-out test + walk-forward).
    "promote_min_tstat": 2.0,
    "promote_min_wf_positive_share": 0.67,
    "cost_sensitivity_pct": [0.0010, 0.0016, 0.0020],
    # Model capacity. Deliberately regularised: ~90k noisy rows, overlapping labels.
    "n_estimators": 400,
    "max_depth": 6,
    "learning_rate": 0.03,
    "min_leaf": 200,
    "threads_per_model": 2,
    "ensemble": ["LR", "RF", "XGB", "LGBM", "CatBoost"],
}


def apply_resolution_config(cfg: Dict[str, Any]) -> None:
    """Derive bar minutes, horizon wall-clock, purge, and export_dir from resolution."""
    res = str(cfg["resolution"])
    if res not in RESOLUTION_MINUTES:
        raise ValueError(
            f"Unsupported resolution {res!r}; use one of {sorted(RESOLUTION_MINUTES)}"
        )
    bar_min = int(RESOLUTION_MINUTES[res])
    horizon_bars = int(cfg["horizon_bars"])
    if bar_min <= 0 or horizon_bars <= 0:
        raise ValueError(
            f"horizon_bars={horizon_bars} and {res} bar minutes ({bar_min}) must be positive"
        )
    cfg["bar_minutes"] = bar_min
    cfg["horizon_bars"] = horizon_bars
    cfg["horizon_minutes"] = horizon_bars * bar_min
    # Purge must cover the label look-ahead so no training label peeks into val.
    cfg["purge_bars"] = horizon_bars + 1
    cfg["export_dir"] = f"export/opportunity_{res}_h{horizon_bars}"


_override = Path("opportunity_run_config.json")
if _override.is_file():
    CONFIG.update(json.loads(_override.read_text(encoding="utf-8")))
apply_resolution_config(CONFIG)

LABEL_NAMES: Tuple[str, ...] = ("LONG", "SHORT", "NO_TRADE")
LABEL_LONG, LABEL_SHORT, LABEL_NO_TRADE = 0, 1, 2
ENSEMBLE_NAME = "ENSEMBLE"
print(json.dumps(CONFIG, indent=2))


{
  "symbol": "BTCUSD",
  "base_url": "https://api.india.delta.exchange",
  "history_days": 900,
  "resolution": "2h",
  "horizon_bars": 48,
  "seed": 42,
  "tp_atr_mult": 4.0,
  "sl_atr_mult": 2.0,
  "atr_period": 14,
  "round_trip_cost_pct": 0.0016,
  "min_tp_cost_ratio": 4.0,
  "minority_rate": 0.15,
  "ambiguous_win_max": 0.5,
  "train_fraction": 0.7,
  "val_fraction": 0.15,
  "calib_fraction": 0.15,
  "min_signals": 100,
  "signal_quantiles": [
    0.99,
    0.98,
    0.97,
    0.95,
    0.92,
    0.9,
    0.85,
    0.8
  ],
  "walk_forward_folds": 4,
  "promote_min_tstat": 2.0,
  "promote_min_wf_positive_share": 0.67,
  "cost_sensitivity_pct": [
    0.001,
    0.0016,
    0.002
  ],
  "n_estimators": 400,
  "max_depth": 6,
  "learning_rate": 0.03,
  "min_leaf": 200,
  "threads_per_model": 2,
  "ensemble": [
    "LR",
    "RF",
    "XGB",
    "LGBM",
    "CatBoost"
  ],
  "bar_minutes": 120,
  "horizon_minutes": 5760,
  "purge_bars": 49,
  "export_dir": "export/opportunity_2h_h48"

In [ ]:
# CELL: SEED
def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))


set_seed(int(CONFIG["seed"]))
print("seed", CONFIG["seed"])


seed 42


In [ ]:
# CELL: FETCH
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95
_RESOLUTION_MINUTES = {
    "1m": 1,
    "3m": 3,
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
    "4h": 240,
    "6h": 360,
    "1d": 1440,
}


def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(_RESOLUTION_MINUTES[resolution]) * 60


def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    return [row for row in result if isinstance(row, dict) and "time" in row]


def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None


def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    import requests

    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []


def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")
    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)
    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)
    if not all_rows:
        raise RuntimeError(f"No candle data returned for symbol={symbol}")
    return _rows_to_frame(all_rows)


def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")
    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    report = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'}"
        )
    return report


end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=int(CONFIG["history_days"]))
_resolution = str(CONFIG["resolution"])
# Delta has no native 10m candles; everything else is fetched at its own resolution.
_source_resolution = "5m" if _resolution == "10m" else _resolution
print(
    f"Fetching {CONFIG['symbol']} {_source_resolution} OHLCV for {CONFIG['history_days']}d "
    f"from {CONFIG['base_url']} (target {_resolution})..."
)
raw_bars = fetch_candles(
    str(CONFIG["symbol"]),
    _source_resolution,
    int(start_dt.timestamp()),
    int(end_dt.timestamp()),
    str(CONFIG["base_url"]),
)
ohlcv_report = validate_ohlcv_completeness(
    raw_bars, _source_resolution, symbol=str(CONFIG["symbol"])
)
print(
    f"OHLCV {CONFIG['symbol']} {_source_resolution}: {ohlcv_report['rows']} bars, "
    f"completeness {ohlcv_report['completeness']:.1%}, gaps={ohlcv_report['gaps']}"
)


Fetching BTCUSD 2h OHLCV for 900d from https://api.india.delta.exchange (target 2h)...


OHLCV BTCUSD 2h: 10800 bars, completeness 100.0%, gaps=0


In [ ]:
# CELL: BARS
def build_10m_ohlcv_from_5m(df5m: pd.DataFrame) -> pd.DataFrame:
    """Exchange-aligned 10m bars from two closed 5m candles. Drop a partial last bucket."""
    src = df5m.copy()
    src["time"] = pd.to_datetime(src["time"], utc=True)
    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in src.columns]
    if missing:
        raise ValueError(f"5m OHLCV missing columns: {missing}")
    if src.empty:
        return src.iloc[0:0].copy()
    grouped = src.set_index("time").sort_index()
    agg = grouped.resample("10min", label="left", closed="left").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample("10min", label="left", closed="left").count()
    complete = counts >= 2
    agg = agg.loc[complete].dropna(subset=["open", "high", "low", "close"])
    out = agg.reset_index()
    return out.sort_values("time").reset_index(drop=True)


if str(CONFIG["resolution"]) == "10m":
    bars = build_10m_ohlcv_from_5m(raw_bars)
else:
    bars = raw_bars
if bars is None or getattr(bars, "empty", True):
    raise ValueError(f"No OHLCV bars for {CONFIG['resolution']}")
print(
    f"{CONFIG['resolution']} bars={len(bars)} "
    f"span={bars['time'].iloc[0]} → {bars['time'].iloc[-1]} "
    f"horizon_bars={CONFIG['horizon_bars']} ({CONFIG['horizon_minutes']}m)"
)


2h bars=10800 span=2024-04-25 08:00:00+00:00 → 2026-10-12 06:00:00+00:00 horizon_bars=48 (5760m)


In [ ]:
# CELL: FEATURES
OHLCV_COLS: Tuple[str, ...] = ("time", "open", "high", "low", "close", "volume")


def _wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def add_opportunity_features(df: pd.DataFrame, *, atr_period: int = 14) -> pd.DataFrame:
    """Causal signal-candle features from the selected timeframe series only.

    Everything is expressed in ATR units or as a ratio so the model is scale
    invariant across the 900-day price range. ATR(14) is also stored for the
    label path. Nothing here looks past the signal candle.
    """
    out = df.copy()
    close = out["close"].astype(np.float64)
    high = out["high"].astype(np.float64)
    low = out["low"].astype(np.float64)
    open_ = out["open"].astype(np.float64)
    volume = out["volume"].astype(np.float64)
    prev_close = close.shift(1)
    tr = pd.concat(
        [high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1
    ).max(axis=1)
    atr = tr.rolling(int(atr_period), min_periods=int(atr_period)).mean()
    atr_safe = atr.replace(0.0, np.nan)
    out["atr"] = atr

    feats: Dict[str, pd.Series] = {}
    # Momentum at several lookbacks, in ATR units.
    feats["ret_1"] = close.pct_change()
    for k in (3, 6, 12, 24, 48, 144):
        feats[f"ret_{k}_atr"] = (close - close.shift(k)) / atr_safe
    # Volatility level and regime.
    feats["atr_pct"] = atr / close.replace(0.0, np.nan)
    atr_long = tr.rolling(int(atr_period) * 6, min_periods=int(atr_period) * 6).mean()
    feats["atr_ratio_84"] = atr / atr_long.replace(0.0, np.nan)
    log_ret = np.log(close).diff()
    rv_12 = log_ret.rolling(12, min_periods=12).std()
    rv_48 = log_ret.rolling(48, min_periods=48).std()
    rv_144 = log_ret.rolling(144, min_periods=144).std()
    feats["rv_12_48"] = rv_12 / rv_48.replace(0.0, np.nan)
    feats["rv_48_144"] = rv_48 / rv_144.replace(0.0, np.nan)
    # Oscillators and trend.
    for period in (7, 14, 28):
        feats[f"rsi_{period}"] = _wilder_rsi(close, period=period)
    for span in (20, 50, 200):
        ema = close.ewm(span=span, adjust=False, min_periods=span).mean()
        feats[f"ema_dist_{span}_atr"] = (close - ema) / atr_safe
        feats[f"ema_slope_{span}_atr"] = (ema - ema.shift(6)) / atr_safe
    # Candle anatomy.
    feats["candle_body"] = (close - open_) / atr_safe
    feats["upper_wick"] = (high - np.maximum(close, open_)) / atr_safe
    feats["lower_wick"] = (np.minimum(close, open_) - low) / atr_safe
    feats["range_atr"] = (high - low) / atr_safe
    # Volume anomalies.
    for window in (20, 100):
        vol_mean = volume.rolling(window, min_periods=window).mean()
        vol_std = volume.rolling(window, min_periods=window).std()
        feats[f"volume_z_{window}"] = (volume - vol_mean) / vol_std.replace(0.0, np.nan)
    # Position inside recent ranges and distance to swing extremes.
    for window in (24, 72, 144):
        hh = high.rolling(window, min_periods=window).max()
        ll = low.rolling(window, min_periods=window).min()
        feats[f"range_pos_{window}"] = (close - ll) / (hh - ll).replace(0.0, np.nan)
        feats[f"dist_high_{window}_atr"] = (hh - close) / atr_safe
        feats[f"dist_low_{window}_atr"] = (close - ll) / atr_safe
    # Consecutive same-direction closes (signed).
    sign = np.sign(close.diff()).fillna(0.0)
    groups = (sign != sign.shift()).cumsum()
    feats["streak"] = sign * (sign.groupby(groups).cumcount() + 1)
    # Intraday / weekly seasonality.
    ts = pd.to_datetime(out["time"], utc=True)
    hour = ts.dt.hour + ts.dt.minute / 60.0
    feats["hour_sin"] = np.sin(2.0 * np.pi * hour / 24.0)
    feats["hour_cos"] = np.cos(2.0 * np.pi * hour / 24.0)
    feats["dow_sin"] = np.sin(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    feats["dow_cos"] = np.cos(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    # Short raw history of the last six candle returns (ATR units).
    for lag in range(1, 7):
        feats[f"ret_lag{lag}_atr"] = (close.shift(lag - 1) - close.shift(lag)) / atr_safe
    return pd.concat([out, pd.DataFrame(feats, index=out.index)], axis=1)


featured = add_opportunity_features(bars, atr_period=int(CONFIG["atr_period"]))
FEATURE_COLS: Tuple[str, ...] = tuple(
    c for c in featured.columns if c not in OHLCV_COLS and c != "atr"
)
print(len(FEATURE_COLS), "features")
null_rates = featured[list(FEATURE_COLS)].isna().mean()
print("max feature null rate", round(float(null_rates.max()), 4), "at", null_rates.idxmax())


46 features
max feature null rate 0.019 at ema_slope_200_atr


In [ ]:
# CELL: LABELS
# Opportunity labels on a closed-bar series. Path is bars t+1 .. t+horizon only.
# Vectorised over bars; semantics are identical to walking each path bar by bar.

import numpy as np
import pandas as pd

_EPS = 1e-9
HORIZON_BARS_DEFAULT = 24
ATR_PERIOD_DEFAULT = 14
TP_ATR_MULT_DEFAULT = 4.0
SL_ATR_MULT_DEFAULT = 2.0

TP_FIRST = 0
SL_FIRST = 1
NEITHER = 2
AMBIGUOUS = 3

LABEL_LONG = 0
LABEL_SHORT = 1
LABEL_NO_TRADE = 2
LABEL_UNLABELED = -1
LABEL_NAMES = ("LONG", "SHORT", "NO_TRADE")


def tp_sl_from_path(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: float,
    atr: float,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> int:
    """Walk future highs/lows. Same-bar TP and SL is AMBIGUOUS, not a win."""
    high = np.asarray(fwd_high, dtype=np.float64).ravel()
    low = np.asarray(fwd_low, dtype=np.float64).ravel()
    if high.size == 0 or high.size != low.size:
        return int(NEITHER)
    denom = max(float(atr), _EPS)
    entry_f = float(entry)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry_f - float(tp_mult) * denom
        sl_level = entry_f + float(sl_mult) * denom
        tp_hit = low <= tp_level
        sl_hit = high >= sl_level
    else:
        tp_level = entry_f + float(tp_mult) * denom
        sl_level = entry_f - float(sl_mult) * denom
        tp_hit = high >= tp_level
        sl_hit = low <= sl_level
    for j in range(high.size):
        hit_tp = bool(tp_hit[j])
        hit_sl = bool(sl_hit[j])
        if hit_tp and hit_sl:
            return int(AMBIGUOUS)
        if hit_tp:
            return int(TP_FIRST)
        if hit_sl:
            return int(SL_FIRST)
    return int(NEITHER)


def _forward_stack(values: np.ndarray, k: int) -> np.ndarray:
    """(k, n) array where row j-1 holds values shifted back by j bars (NaN padded)."""
    n = values.shape[0]
    out = np.full((k, n), np.nan, dtype=np.float64)
    for j in range(1, k + 1):
        if n - j > 0:
            out[j - 1, : n - j] = values[j:]
    return out


def _side_codes_vectorised(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: np.ndarray,
    atr: np.ndarray,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> np.ndarray:
    """Vectorised twin of ``tp_sl_from_path`` for every bar at once."""
    denom = np.maximum(atr, _EPS)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry - float(tp_mult) * denom
        sl_level = entry + float(sl_mult) * denom
        tp_hits = fwd_low <= tp_level[None, :]
        sl_hits = fwd_high >= sl_level[None, :]
    else:
        tp_level = entry + float(tp_mult) * denom
        sl_level = entry - float(sl_mult) * denom
        tp_hits = fwd_high >= tp_level[None, :]
        sl_hits = fwd_low <= sl_level[None, :]
    any_hit = tp_hits | sl_hits
    has_hit = any_hit.any(axis=0)
    first = np.argmax(any_hit, axis=0)
    cols = np.arange(entry.shape[0])
    tp_first = tp_hits[first, cols]
    sl_first = sl_hits[first, cols]
    codes = np.where(
        ~has_hit,
        NEITHER,
        np.where(tp_first & sl_first, AMBIGUOUS, np.where(tp_first, TP_FIRST, SL_FIRST)),
    )
    return codes.astype(np.int64)


def _causal_atr(df: pd.DataFrame, period: int = ATR_PERIOD_DEFAULT) -> np.ndarray:
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]])
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    return pd.Series(tr).rolling(int(period), min_periods=int(period)).mean().to_numpy(
        dtype=np.float64
    )


def compute_opportunity_labels(
    df: pd.DataFrame,
    *,
    tp_mult: float = TP_ATR_MULT_DEFAULT,
    sl_mult: float = SL_ATR_MULT_DEFAULT,
    horizon_bars: int = HORIZON_BARS_DEFAULT,
    atr_period: int = ATR_PERIOD_DEFAULT,
) -> pd.DataFrame:
    """Label each closed bar. Last ``horizon_bars`` rows stay unlabeled.

    Realised P&L columns (``long_pnl_pct`` / ``short_pnl_pct``) are gross of
    cost: +TP distance on TP_FIRST, −SL distance on SL_FIRST **and** AMBIGUOUS
    (a same-bar double touch is scored as a stop), timeout return on NEITHER.
    """
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    atr = np.asarray(_causal_atr(out, period=int(atr_period)), dtype=np.float64)
    if atr.shape[0] != n:
        atr = np.resize(atr, n)

    fwd_high = _forward_stack(high, k)
    fwd_low = _forward_stack(low, k)
    fwd_close_end = np.full(n, np.nan, dtype=np.float64)
    if n - k > 0:
        fwd_close_end[: n - k] = close[k:]

    index = np.arange(n)
    valid = (
        (index < n - k)
        & np.isfinite(atr)
        & (atr > _EPS)
        & np.isfinite(close)
        & (np.abs(close) > _EPS)
    )
    atr_safe = np.where(valid, atr, np.nan)
    entry_safe = np.where(valid, close, np.nan)

    l_code = _side_codes_vectorised(fwd_high, fwd_low, entry_safe, atr_safe, tp_mult, sl_mult, "LONG")
    s_code = _side_codes_vectorised(fwd_high, fwd_low, entry_safe, atr_safe, tp_mult, sl_mult, "SHORT")
    l_win = l_code == TP_FIRST
    s_win = s_code == TP_FIRST
    label = np.where(l_win & ~s_win, LABEL_LONG, np.where(s_win & ~l_win, LABEL_SHORT, LABEL_NO_TRADE))

    with np.errstate(invalid="ignore", divide="ignore"):
        max_high = np.nanmax(np.where(np.isfinite(fwd_high), fwd_high, -np.inf), axis=0)
        min_low = np.nanmin(np.where(np.isfinite(fwd_low), fwd_low, np.inf), axis=0)
        upside = np.maximum((max_high - entry_safe) / atr_safe, 0.0)
        downside = np.maximum((entry_safe - min_low) / atr_safe, 0.0)
        timeout_ret = (fwd_close_end - entry_safe) / entry_safe
        tp_pct = float(tp_mult) * atr_safe / entry_safe
        sl_pct = float(sl_mult) * atr_safe / entry_safe
        long_pnl = np.where(
            l_code == TP_FIRST, tp_pct, np.where(l_code == NEITHER, timeout_ret, -sl_pct)
        )
        short_pnl = np.where(
            s_code == TP_FIRST, tp_pct, np.where(s_code == NEITHER, -timeout_ret, -sl_pct)
        )

    def _masked(values: np.ndarray) -> np.ndarray:
        return np.where(valid, values, np.nan)

    out["atr"] = atr
    out["trade_label"] = np.where(valid, label, LABEL_UNLABELED).astype(np.int64)
    out["long_result"] = _masked(l_win.astype(np.float64))
    out["short_result"] = _masked(s_win.astype(np.float64))
    out["long_code"] = _masked(l_code.astype(np.float64))
    out["short_code"] = _masked(s_code.astype(np.float64))
    out["upside"] = _masked(upside)
    out["downside"] = _masked(downside)
    out["timeout_ret"] = _masked(timeout_ret)
    out["tp_pct"] = _masked(tp_pct)
    out["sl_pct"] = _masked(sl_pct)
    out["long_pnl_pct"] = _masked(long_pnl)
    out["short_pnl_pct"] = _masked(short_pnl)
    names = np.array(["UNLABELED"] * n, dtype=object)
    names[out["trade_label"].to_numpy() == LABEL_LONG] = "LONG"
    names[out["trade_label"].to_numpy() == LABEL_SHORT] = "SHORT"
    names[out["trade_label"].to_numpy() == LABEL_NO_TRADE] = "NO_TRADE"
    out["trade_name"] = names
    return out


In [ ]:
# CELL: APPLY_LABELS
labeled = compute_opportunity_labels(
    featured,
    tp_mult=float(CONFIG["tp_atr_mult"]),
    sl_mult=float(CONFIG["sl_atr_mult"]),
    horizon_bars=int(CONFIG["horizon_bars"]),
    atr_period=int(CONFIG["atr_period"]),
)
valid = labeled["trade_label"].to_numpy() >= 0
print(f"labeled rows={int(valid.sum())} / {len(labeled)}")
print(labeled.loc[valid, "trade_name"].value_counts(normalize=True).round(4))
print(
    "median TP distance",
    f"{labeled.loc[valid, 'tp_pct'].median():.4%}",
    "median SL distance",
    f"{labeled.loc[valid, 'sl_pct'].median():.4%}",
    "round trip cost",
    f"{float(CONFIG['round_trip_cost_pct']):.4%}",
)


labeled rows=10739 / 10800
trade_name
NO_TRADE    0.4223
SHORT       0.2911
LONG        0.2866
Name: proportion, dtype: float64
median TP distance 3.5239% median SL distance 1.7619% round trip cost 0.1600%


In [ ]:
# CELL: GATE
def train_period_mask(n: int, train_fraction: float) -> np.ndarray:
    cut = int(n * float(train_fraction))
    mask = np.zeros(n, dtype=bool)
    mask[:cut] = True
    return mask


def _path_code_counts(codes: np.ndarray, usable: np.ndarray) -> Dict[str, int]:
    finite = usable & np.isfinite(codes)
    return {
        "TP_FIRST": int(np.sum(finite & (codes == TP_FIRST))),
        "SL_FIRST": int(np.sum(finite & (codes == SL_FIRST))),
        "NEITHER": int(np.sum(finite & (codes == NEITHER))),
        "AMBIGUOUS": int(np.sum(finite & (codes == AMBIGUOUS))),
    }


def summarize_label_gate(
    df: pd.DataFrame,
    *,
    train_fraction: float,
    round_trip_cost_pct: float,
    minority_rate: float,
    ambiguous_win_max: float,
    min_tp_cost_ratio: float = 4.0,
) -> Dict[str, Any]:
    """Decide whether the bracket is worth training on, using train rows only.

    Besides class mix and oracle exclusive expectancy, the gate checks the
    bracket economics: the median take-profit distance must be at least
    ``min_tp_cost_ratio`` times the round-trip cost, and the implied
    break-even win rate ``(sl + cost) / (tp + sl)`` is reported.
    """
    n = len(df)
    in_train = train_period_mask(n, train_fraction)
    labels = df["trade_label"].to_numpy(dtype=np.int64)
    usable = in_train & (labels >= 0)
    n_use = int(usable.sum())
    if n_use == 0:
        raise RuntimeError("Train-period gate has no labeled rows")
    mix = {}
    mix_ok = True
    for cid, name in enumerate(LABEL_NAMES):
        rate = float(np.mean(labels[usable] == cid))
        mix[name] = rate
        if rate < float(minority_rate):
            mix_ok = False
    long_code = df["long_code"].to_numpy(dtype=np.float64)
    short_code = df["short_code"].to_numpy(dtype=np.float64)
    cost = float(round_trip_cost_pct)
    long_pnl = df["long_pnl_pct"].to_numpy(dtype=np.float64) - cost
    short_pnl = df["short_pnl_pct"].to_numpy(dtype=np.float64) - cost

    always_long_mask = usable & np.isfinite(long_pnl)
    always_short_mask = usable & np.isfinite(short_pnl)
    always_in_long = (
        float(np.mean(long_pnl[always_long_mask])) if always_long_mask.any() else float("nan")
    )
    always_in_short = (
        float(np.mean(short_pnl[always_short_mask])) if always_short_mask.any() else float("nan")
    )

    excl_long_mask = usable & (labels == LABEL_LONG) & np.isfinite(long_pnl)
    excl_short_mask = usable & (labels == LABEL_SHORT) & np.isfinite(short_pnl)
    exclusive_long = (
        float(np.mean(long_pnl[excl_long_mask])) if excl_long_mask.any() else float("nan")
    )
    exclusive_short = (
        float(np.mean(short_pnl[excl_short_mask])) if excl_short_mask.any() else float("nan")
    )

    def _ambig_share(codes: np.ndarray) -> float:
        finite = usable & np.isfinite(codes)
        tp = finite & (codes == TP_FIRST)
        amb = finite & (codes == AMBIGUOUS)
        denom = int(tp.sum() + amb.sum())
        if denom == 0:
            return 0.0
        return float(amb.sum()) / float(denom)

    long_ambig = _ambig_share(long_code)
    short_ambig = _ambig_share(short_code)

    if "tp_pct" in df.columns and "sl_pct" in df.columns:
        tp_pct = df["tp_pct"].to_numpy(dtype=np.float64)
        sl_pct = df["sl_pct"].to_numpy(dtype=np.float64)
    else:
        atr_pct = df["atr"].to_numpy(dtype=np.float64) / df["close"].to_numpy(dtype=np.float64)
        tp_pct = float(CONFIG["tp_atr_mult"]) * atr_pct
        sl_pct = float(CONFIG["sl_atr_mult"]) * atr_pct
    econ_mask = usable & np.isfinite(tp_pct) & np.isfinite(sl_pct)
    median_tp = float(np.median(tp_pct[econ_mask])) if econ_mask.any() else float("nan")
    median_sl = float(np.median(sl_pct[econ_mask])) if econ_mask.any() else float("nan")
    tp_cost_ratio = median_tp / cost if cost > 0 and np.isfinite(median_tp) else float("inf")
    breakeven_win_rate = (
        (median_sl + cost) / (median_tp + median_sl)
        if np.isfinite(median_tp) and np.isfinite(median_sl) and (median_tp + median_sl) > 0
        else float("nan")
    )
    long_tp_rate = float(np.mean(long_code[usable & np.isfinite(long_code)] == TP_FIRST))
    short_tp_rate = float(np.mean(short_code[usable & np.isfinite(short_code)] == TP_FIRST))

    reasons: List[str] = []
    if not mix_ok:
        reasons.append("class_mix_below_floor")
    if not (np.isfinite(exclusive_long) and exclusive_long > 0.0):
        reasons.append("exclusive_long_expectancy_not_positive")
    if not (np.isfinite(exclusive_short) and exclusive_short > 0.0):
        reasons.append("exclusive_short_expectancy_not_positive")
    if long_ambig >= float(ambiguous_win_max):
        reasons.append("long_ambiguous_majority")
    if short_ambig >= float(ambiguous_win_max):
        reasons.append("short_ambiguous_majority")
    if tp_cost_ratio < float(min_tp_cost_ratio):
        reasons.append("tp_distance_below_cost_ratio")
    passed = len(reasons) == 0
    return {
        "passed": passed,
        "n_train_labeled": n_use,
        "class_mix": mix,
        "minority_rate": float(minority_rate),
        "exclusive_long_expectancy_pct": exclusive_long,
        "exclusive_short_expectancy_pct": exclusive_short,
        "always_in_long_expectancy_pct": always_in_long,
        "always_in_short_expectancy_pct": always_in_short,
        "round_trip_cost_pct": cost,
        "median_tp_pct": median_tp,
        "median_sl_pct": median_sl,
        "tp_cost_ratio": tp_cost_ratio,
        "min_tp_cost_ratio": float(min_tp_cost_ratio),
        "breakeven_win_rate": breakeven_win_rate,
        "base_long_tp_rate": long_tp_rate,
        "base_short_tp_rate": short_tp_rate,
        "long_ambiguous_share": long_ambig,
        "short_ambiguous_share": short_ambig,
        "long_path_codes": _path_code_counts(long_code, usable),
        "short_path_codes": _path_code_counts(short_code, usable),
        "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
        "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
        "reasons": reasons,
    }


EXPORT_DIR = Path(str(CONFIG["export_dir"]))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
label_report = summarize_label_gate(
    labeled,
    train_fraction=float(CONFIG["train_fraction"]),
    round_trip_cost_pct=float(CONFIG["round_trip_cost_pct"]),
    minority_rate=float(CONFIG["minority_rate"]),
    ambiguous_win_max=float(CONFIG["ambiguous_win_max"]),
    min_tp_cost_ratio=float(CONFIG["min_tp_cost_ratio"]),
)
report_path = EXPORT_DIR / "label_report.json"
report_path.write_text(json.dumps(label_report, indent=2), encoding="utf-8")
print(json.dumps(label_report, indent=2))
print("wrote", report_path)
if not label_report["passed"]:
    raise RuntimeError(
        "Train-period label gate failed: "
        f"{label_report['reasons']}. Change CONFIG tp/sl ATR multiples or horizon and "
        "re-run from the top. Do not fit models."
    )
print(
    f"Gate passed. Break-even win rate {label_report['breakeven_win_rate']:.1%} vs base "
    f"TP rates long {label_report['base_long_tp_rate']:.1%} / short "
    f"{label_report['base_short_tp_rate']:.1%}. Training may start."
)


{
  "passed": true,
  "n_train_labeled": 7546,
  "class_mix": {
    "LONG": 0.29326795653326265,
    "SHORT": 0.28650940895838856,
    "NO_TRADE": 0.4202226345083488
  },
  "minority_rate": 0.15,
  "exclusive_long_expectancy_pct": 0.03391622462358931,
  "exclusive_short_expectancy_pct": 0.03450071424857617,
  "always_in_long_expectancy_pct": -0.00019954519922331065,
  "always_in_short_expectancy_pct": -0.0019828182009252898,
  "round_trip_cost_pct": 0.0016,
  "median_tp_pct": 0.03585976446897242,
  "median_sl_pct": 0.01792988223448621,
  "tp_cost_ratio": 22.41235279310776,
  "min_tp_cost_ratio": 4.0,
  "breakeven_win_rate": 0.3630788345228238,
  "base_long_tp_rate": 0.29326795653326265,
  "base_short_tp_rate": 0.28650940895838856,
  "long_ambiguous_share": 0.0009029345372460496,
  "short_ambiguous_share": 0.0004623208506703652,
  "long_path_codes": {
    "TP_FIRST": 2213,
    "SL_FIRST": 4345,
    "NEITHER": 986,
    "AMBIGUOUS": 2
  },
  "short_path_codes": {
    "TP_FIRST": 2162,
   

In [ ]:
# CELL: SPLIT
def purged_index_splits(
    n: int,
    *,
    train_fraction: float,
    val_fraction: float,
    purge_bars: int,
) -> Dict[str, np.ndarray]:
    """Time-ordered split. Purge ``purge_bars`` on each side of every cut."""
    n = int(n)
    purge = int(purge_bars)
    train_end = int(n * float(train_fraction))
    val_len = int(n * float(val_fraction))
    train_idx = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
    val_start = min(train_end + purge, n)
    val_end = min(val_start + val_len, n)
    val_idx = np.arange(val_start, max(val_end - purge, val_start), dtype=np.int64)
    test_start = min(val_end + purge, n)
    test_idx = np.arange(test_start, n, dtype=np.int64)
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def development_walk_forward(
    n_dev: int,
    *,
    folds: int = 4,
    purge_bars: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Expanding-window folds inside the development prefix only."""
    folds = max(2, int(folds))
    purge = int(purge_bars)
    edges = [int(n_dev * (i + 1) / folds) for i in range(folds)]
    out: List[Tuple[np.ndarray, np.ndarray]] = []
    for i in range(1, folds):
        train_end = edges[i - 1]
        val_end = edges[i]
        tr = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
        va = np.arange(min(train_end + purge, n_dev), val_end, dtype=np.int64)
        if len(tr) == 0 or len(va) == 0:
            continue
        out.append((tr, va))
    return out


def inner_calibration_split(
    train_idx: np.ndarray, *, calib_fraction: float, purge_bars: int
) -> Tuple[np.ndarray, np.ndarray]:
    """Reserve the purged tail of a training window for threshold selection."""
    n = len(train_idx)
    cut = int(n * (1.0 - float(calib_fraction)))
    fit_idx = train_idx[: max(cut - int(purge_bars), 0)]
    calib_idx = train_idx[cut:]
    return fit_idx, calib_idx


splits = purged_index_splits(
    len(labeled),
    train_fraction=float(CONFIG["train_fraction"]),
    val_fraction=float(CONFIG["val_fraction"]),
    purge_bars=int(CONFIG["purge_bars"]),
)
for key, idx in splits.items():
    print(key, "n", len(idx), "range", int(idx[0]) if len(idx) else None, int(idx[-1]) if len(idx) else None)
dev_end = int(splits["test"][0]) if len(splits["test"]) else len(labeled)
wf_folds = development_walk_forward(
    dev_end, folds=int(CONFIG["walk_forward_folds"]), purge_bars=int(CONFIG["purge_bars"])
)
print("walk-forward folds", len(wf_folds), "dev_end", dev_end)


train n 7510 range 0 7509
val n 1571 range 7608 9178
test n 1523 range 9277 10799
walk-forward folds 3 dev_end 9277


In [ ]:
# CELL: DATASET
def build_rows(
    frame: pd.DataFrame, *, feature_cols: Sequence[str]
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """One tabular row per labeled bar with finite features and finite path targets."""
    feats = frame.loc[:, list(feature_cols)].to_numpy(dtype=np.float64)
    y_cls = frame["trade_label"].to_numpy(dtype=np.int64)
    y_reg = frame.loc[:, ["upside", "downside"]].to_numpy(dtype=np.float64)
    ok = (y_cls >= 0) & np.isfinite(feats).all(axis=1) & np.isfinite(y_reg).all(axis=1)
    rows = np.flatnonzero(ok).astype(np.int64)
    return (
        feats[rows].astype(np.float32),
        y_cls[rows],
        y_reg[rows].astype(np.float32),
        rows,
    )


all_x, all_y, all_r, all_rows = build_rows(labeled, feature_cols=FEATURE_COLS)
print(
    "rows",
    all_x.shape,
    "class mix",
    {n: round(float(np.mean(all_y == i)), 4) for i, n in enumerate(LABEL_NAMES)},
)


def rows_in(split_idx: np.ndarray) -> np.ndarray:
    """Boolean selector over ``all_rows`` for a set of frame indices."""
    allowed = np.zeros(len(labeled), dtype=bool)
    allowed[np.asarray(split_idx, dtype=np.int64)] = True
    return allowed[all_rows]


MIN_SIGNAL_TP_PCT = float(CONFIG["min_tp_cost_ratio"]) * float(CONFIG["round_trip_cost_pct"])


def signal_eligible(rows: np.ndarray) -> np.ndarray:
    """Quiet-market guard: a candle may carry a signal only if its TP distance pays for cost."""
    tp_pct = labeled["tp_pct"].to_numpy(dtype=np.float64)[rows]
    return np.isfinite(tp_pct) & (tp_pct >= MIN_SIGNAL_TP_PCT)


print(
    f"signal eligibility: TP distance >= {MIN_SIGNAL_TP_PCT:.4%} -> "
    f"{float(signal_eligible(all_rows).mean()):.1%} of labeled candles"
)


def fit_scaler(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def scale_x(x: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((x - mean) / std).astype(np.float32)


sel_train = rows_in(splits["train"])
sel_val = rows_in(splits["val"])
sel_test = rows_in(splits["test"])
feat_mean, feat_std = fit_scaler(all_x[sel_train])
reg_mean = all_r[sel_train].mean(axis=0, keepdims=True)
reg_std = all_r[sel_train].std(axis=0, keepdims=True)
reg_std = np.where(reg_std < 1e-6, 1.0, reg_std)


def scale_r(r: np.ndarray) -> np.ndarray:
    return ((r - reg_mean) / reg_std).astype(np.float32)


X_train = scale_x(all_x[sel_train], feat_mean, feat_std)
X_val = scale_x(all_x[sel_val], feat_mean, feat_std)
X_test = scale_x(all_x[sel_test], feat_mean, feat_std)
y_train, y_val, y_test = all_y[sel_train], all_y[sel_val], all_y[sel_test]
r_train, r_val, r_test = all_r[sel_train], all_r[sel_val], all_r[sel_test]
R_train, R_val, R_test = scale_r(r_train), scale_r(r_val), scale_r(r_test)
print("train", X_train.shape, "val", X_val.shape, "test", X_test.shape)


rows (10547, 46) class mix {'LONG': 0.2869, 'SHORT': 0.2916, 'NO_TRADE': 0.4215}
signal eligibility: TP distance >= 0.6400% -> 100.0% of labeled candles
train (7305, 46) val (1571, 46) test (1475, 46)


In [ ]:
# CELL: MODEL
def make_family(name: str, seed: int) -> Dict[str, Any]:
    """Return unfitted classifier + two regressors for one family.

    No class re-weighting anywhere: the decision rule thresholds calibrated-ish
    probabilities, and ``balanced`` weights distort them.
    """
    est = int(CONFIG["n_estimators"])
    depth = int(CONFIG["max_depth"])
    lr = float(CONFIG["learning_rate"])
    leaf = int(CONFIG["min_leaf"])
    threads = int(CONFIG["threads_per_model"])
    if name == "LR":
        clf = LogisticRegression(max_iter=3000, C=0.1, solver="lbfgs")
        up = Ridge(alpha=10.0)
        dn = Ridge(alpha=10.0)
    elif name == "RF":
        rf_kwargs = dict(
            n_estimators=est,
            max_depth=depth + 2,
            min_samples_leaf=leaf,
            max_features="sqrt",
            n_jobs=threads,
        )
        clf = RandomForestClassifier(random_state=seed, **rf_kwargs)
        up = RandomForestRegressor(random_state=seed, **rf_kwargs)
        dn = RandomForestRegressor(random_state=seed + 1, **rf_kwargs)
    elif name == "XGB":
        xgb_kwargs = dict(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_weight=leaf,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            tree_method="hist",
            n_jobs=threads,
            verbosity=0,
        )
        clf = XGBClassifier(
            objective="multi:softprob", eval_metric="mlogloss", random_state=seed, **xgb_kwargs
        )
        up = XGBRegressor(random_state=seed, **xgb_kwargs)
        dn = XGBRegressor(random_state=seed + 1, **xgb_kwargs)
    elif name == "LGBM":
        lgb_kwargs = dict(
            n_estimators=est,
            num_leaves=31,
            max_depth=-1,
            learning_rate=lr,
            min_child_samples=leaf,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            n_jobs=threads,
            verbosity=-1,
        )
        clf = LGBMClassifier(random_state=seed, **lgb_kwargs)
        up = LGBMRegressor(random_state=seed, **lgb_kwargs)
        dn = LGBMRegressor(random_state=seed + 1, **lgb_kwargs)
    elif name == "CatBoost":
        cb_kwargs = dict(
            iterations=est,
            depth=depth,
            learning_rate=lr,
            l2_leaf_reg=10.0,
            verbose=False,
            thread_count=threads,
            allow_writing_files=False,
        )
        clf = CatBoostClassifier(loss_function="MultiClass", random_seed=seed, **cb_kwargs)
        up = CatBoostRegressor(random_seed=seed, **cb_kwargs)
        dn = CatBoostRegressor(random_seed=seed + 1, **cb_kwargs)
    else:
        raise ValueError(name)
    return {"name": name, "clf": clf, "up": up, "dn": dn}


print("families", CONFIG["ensemble"], "+", ENSEMBLE_NAME)


families ['LR', 'RF', 'XGB', 'LGBM', 'CatBoost'] + ENSEMBLE


In [ ]:
# CELL: TRAIN
def confusion(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 3) -> np.ndarray:
    mat = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y_true.tolist(), y_pred.tolist()):
        if 0 <= int(t) < n_classes and 0 <= int(p) < n_classes:
            mat[int(t), int(p)] += 1
    return mat


def class_precision(mat: np.ndarray, cls: int) -> float:
    col = mat[:, cls]
    denom = float(col.sum())
    if denom <= 0:
        return float("nan")
    return float(mat[cls, cls]) / denom


def balanced_accuracy(mat: np.ndarray) -> float:
    recalls = []
    for i in range(mat.shape[0]):
        denom = float(mat[i].sum())
        recalls.append(float(mat[i, i]) / denom if denom else float("nan"))
    finite = [r for r in recalls if np.isfinite(r)]
    return float(np.mean(finite)) if finite else float("nan")


def corr(a: np.ndarray, b: np.ndarray) -> float:
    if a.size < 3:
        return float("nan")
    if float(np.std(a)) < 1e-12 or float(np.std(b)) < 1e-12:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def as_cls(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr).reshape(-1).astype(np.int64)


def as_1d(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr, dtype=np.float64).reshape(-1)


def proba_of(clf: Any, X: np.ndarray) -> np.ndarray:
    """(n, 3) class probabilities in LONG/SHORT/NO_TRADE order regardless of estimator."""
    p = np.asarray(clf.predict_proba(X), dtype=np.float64)
    if p.ndim == 1:
        p = p.reshape(len(X), -1)
    classes = getattr(clf, "classes_", None)
    if classes is None:
        return p
    full = np.zeros((p.shape[0], len(LABEL_NAMES)), dtype=np.float64)
    for j, c in enumerate(np.asarray(classes).reshape(-1).astype(int)):
        full[:, c] = p[:, j]
    return full


def classifier_report(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, Any]:
    cm = confusion(y_true, y_pred)
    return {
        "precision_LONG": class_precision(cm, LABEL_LONG),
        "precision_SHORT": class_precision(cm, LABEL_SHORT),
        "n_LONG": float(np.sum(y_pred == LABEL_LONG)),
        "n_SHORT": float(np.sum(y_pred == LABEL_SHORT)),
        "balanced_accuracy": balanced_accuracy(cm),
        "confusion": cm.tolist(),
    }


_PNL_COL = {LABEL_LONG: "long_pnl_pct", LABEL_SHORT: "short_pnl_pct"}
_SIDE_NAME = {LABEL_LONG: "LONG", LABEL_SHORT: "SHORT"}


def net_pnl(cls: int, rows: np.ndarray, cost: Optional[float] = None) -> np.ndarray:
    """Net-of-cost realised P&L (fraction of price) for taking ``cls`` on ``rows``."""
    c = float(CONFIG["round_trip_cost_pct"]) if cost is None else float(cost)
    return labeled[_PNL_COL[int(cls)]].to_numpy(dtype=np.float64)[rows] - c


def signal_expectancy(
    pred_cls: np.ndarray, rows: np.ndarray, cost: Optional[float] = None
) -> Dict[str, float]:
    """Per-side expectancy, t-stat, profit factor and total for the predicted signals."""
    out: Dict[str, float] = {}
    for cls, name in _SIDE_NAME.items():
        take = pred_cls == cls
        pnl = net_pnl(cls, rows[take], cost) if take.any() else np.zeros(0)
        pnl = pnl[np.isfinite(pnl)]
        n = int(pnl.size)
        out[f"{name}_n"] = float(n)
        out[f"{name}_expectancy"] = float(pnl.mean()) if n else float("nan")
        std = float(pnl.std(ddof=1)) if n > 1 else float("nan")
        out[f"{name}_tstat"] = (
            float(pnl.mean() / (std / math.sqrt(n))) if n > 1 and std > 0 else float("nan")
        )
        wins = float(pnl[pnl > 0].sum())
        losses = float(-pnl[pnl < 0].sum())
        out[f"{name}_profit_factor"] = (
            wins / losses if losses > 0 else (float("inf") if wins > 0 else float("nan"))
        )
        out[f"{name}_total_pct"] = float(pnl.sum())
    return out


def expectancy_rank_score(exp_report: Dict[str, float]) -> float:
    """Mean expectancy over sides with >= ``min_signals`` trades; -inf when none qualify."""
    min_n = int(CONFIG["min_signals"])
    parts: List[float] = []
    for name in ("LONG", "SHORT"):
        n = float(exp_report.get(f"{name}_n", 0.0))
        e = float(exp_report.get(f"{name}_expectancy", float("nan")))
        if n >= min_n and np.isfinite(e):
            parts.append(e)
    return float(np.mean(parts)) if parts else float("-inf")


def choose_thresholds(proba: np.ndarray, rows: np.ndarray) -> Dict[str, Optional[float]]:
    """Per side, the probability threshold that maximises calibration expectancy.

    Candidates are quantiles of the side's probability on the calibration rows.
    A side with no candidate reaching ``min_signals`` trades is disabled (None).
    """
    min_n = int(CONFIG["min_signals"])
    eligible = signal_eligible(rows)
    out: Dict[str, Optional[float]] = {}
    for cls, name in _SIDE_NAME.items():
        other = LABEL_SHORT if cls == LABEL_LONG else LABEL_LONG
        p_side = proba[:, cls]
        p_other = proba[:, other]
        pnl_all = net_pnl(cls, rows)
        best_thr: Optional[float] = None
        best_e = float("-inf")
        if not eligible.any():
            out[name] = None
            continue
        for q in CONFIG["signal_quantiles"]:
            thr = float(np.quantile(p_side[eligible], float(q)))
            take = eligible & (p_side >= thr) & (p_side > p_other) & np.isfinite(pnl_all)
            n = int(take.sum())
            if n < min_n:
                continue
            e = float(pnl_all[take].mean())
            if e > best_e:
                best_thr, best_e = thr, e
        out[name] = best_thr
    return out


def apply_thresholds(
    proba: np.ndarray, thr: Dict[str, Optional[float]], eligible: Optional[np.ndarray] = None
) -> np.ndarray:
    """Live decision rule.

    Abstain unless the candle is eligible (TP distance pays for cost), P(side)
    clears the side's threshold and beats the other side.
    """
    pred = np.full(proba.shape[0], LABEL_NO_TRADE, dtype=np.int64)
    ok = np.ones(proba.shape[0], dtype=bool) if eligible is None else np.asarray(eligible, dtype=bool)
    p_long, p_short = proba[:, LABEL_LONG], proba[:, LABEL_SHORT]
    if thr.get("LONG") is not None:
        pred[ok & (p_long >= float(thr["LONG"])) & (p_long > p_short)] = LABEL_LONG
    if thr.get("SHORT") is not None:
        pred[ok & (p_short >= float(thr["SHORT"])) & (p_short > p_long)] = LABEL_SHORT
    return pred


def score_probas(
    p_calib: np.ndarray,
    rows_calib: np.ndarray,
    p_eval: np.ndarray,
    rows_eval: np.ndarray,
    y_eval: np.ndarray,
) -> Dict[str, Any]:
    thr = choose_thresholds(p_calib, rows_calib)
    pred = apply_thresholds(p_eval, thr, signal_eligible(rows_eval))
    exp_rep = signal_expectancy(pred, rows_eval)
    cls_rep = classifier_report(y_eval, pred)
    return {
        "thresholds": thr,
        "cls": {k: v for k, v in cls_rep.items() if k != "confusion"},
        "confusion": cls_rep["confusion"],
        "exp": exp_rep,
        "score": expectancy_rank_score(exp_rep),
        "pred": pred,
    }


def fit_fold(
    name: str, fold: str, fit_sel: np.ndarray, calib_sel: np.ndarray, eval_sel: np.ndarray, *, seed: int
) -> Dict[str, Any]:
    """Fit one family on ``fit_sel`` rows, return probabilities on calibration and eval rows."""
    mean, std = fit_scaler(all_x[fit_sel])
    pack = make_family(name, seed)
    pack["clf"].fit(scale_x(all_x[fit_sel], mean, std), all_y[fit_sel])
    result = {
        "name": name,
        "fold": fold,
        "clf": pack["clf"],
        "scaler": (mean, std),
        "p_calib": proba_of(pack["clf"], scale_x(all_x[calib_sel], mean, std)),
        "p_eval": proba_of(pack["clf"], scale_x(all_x[eval_sel], mean, std)),
    }
    if fold == "final":
        pack["up"].fit(X_train, R_train[:, 0])
        pack["dn"].fit(X_train, R_train[:, 1])
        result["up"] = pack["up"]
        result["dn"] = pack["dn"]
    return result


# Fold plan: expanding walk-forward folds inside development, then the final split.
# Every fold has disjoint fit / calibration / evaluation rows; thresholds are chosen
# on calibration rows only, so evaluation expectancy is out-of-sample for both the
# model and the decision rule.
fold_plan: List[Tuple[str, np.ndarray, np.ndarray, np.ndarray]] = []
for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
    fit_idx, calib_idx = inner_calibration_split(
        tr_idx, calib_fraction=float(CONFIG["calib_fraction"]), purge_bars=int(CONFIG["purge_bars"])
    )
    fold_plan.append((f"wf{fold_i}", rows_in(fit_idx), rows_in(calib_idx), rows_in(va_idx)))
fold_plan.append(("final", sel_train, sel_val, sel_test))
for fold, f_sel, c_sel, e_sel in fold_plan:
    print(f"{fold}: fit={int(f_sel.sum())} calib={int(c_sel.sum())} eval={int(e_sel.sum())}")

FAMILIES: List[str] = list(CONFIG["ensemble"])
runs: Dict[str, Dict[str, Dict[str, Any]]] = {fold: {} for fold, *_ in fold_plan}
print("Fitting", FAMILIES, "x", [f for f, *_ in fold_plan], "in parallel...")
t_start = time.time()
with ThreadPoolExecutor(max_workers=len(FAMILIES)) as pool:
    futs = {}
    for fold_i, (fold, f_sel, c_sel, e_sel) in enumerate(fold_plan):
        for name in FAMILIES:
            fut = pool.submit(
                fit_fold, name, fold, f_sel, c_sel, e_sel, seed=int(CONFIG["seed"]) + fold_i
            )
            futs[fut] = (fold, name)
    for fut in as_completed(futs):
        fold, name = futs[fut]
        try:
            runs[fold][name] = fut.result()
            print(f"  done {name:9s} {fold}  ({time.time() - t_start:.0f}s)")
        except Exception as exc:  # keep the bake-off alive if one family fails
            print(f"  FAILED {name} {fold}: {exc!r}")

# Soft-vote ensemble over the members that succeeded in a fold.
for fold, f_sel, c_sel, e_sel in fold_plan:
    members = [runs[fold][n] for n in FAMILIES if n in runs[fold]]
    if len(members) >= 2:
        runs[fold][ENSEMBLE_NAME] = {
            "name": ENSEMBLE_NAME,
            "fold": fold,
            "members": [m["name"] for m in members],
            "p_calib": np.mean([m["p_calib"] for m in members], axis=0),
            "p_eval": np.mean([m["p_eval"] for m in members], axis=0),
        }

CANDIDATES: List[str] = [n for n in FAMILIES + [ENSEMBLE_NAME] if n in runs["final"]]
scores: Dict[str, Dict[str, Dict[str, Any]]] = {name: {} for name in CANDIDATES}
for fold, f_sel, c_sel, e_sel in fold_plan:
    rows_c, rows_e, y_e = all_rows[c_sel], all_rows[e_sel], all_y[e_sel]
    for name in CANDIDATES:
        if name not in runs[fold]:
            continue
        r = runs[fold][name]
        scores[name][fold] = score_probas(r["p_calib"], rows_c, r["p_eval"], rows_e, y_e)

WF_FOLDS: List[str] = [fold for fold, *_ in fold_plan if fold != "final"]


def walk_forward_summary(name: str) -> Dict[str, float]:
    vals = [scores[name][f]["score"] for f in WF_FOLDS if f in scores[name]]
    finite = [v for v in vals if np.isfinite(v)]
    return {
        "wf_folds_scored": float(len(finite)),
        "wf_mean_score": float(np.mean(finite)) if finite else float("-inf"),
        "wf_positive_share": float(np.mean([v > 0 for v in finite])) if finite else 0.0,
        "wf_min_score": float(np.min(finite)) if finite else float("-inf"),
    }


leaderboard: Dict[str, Dict[str, Any]] = {}
for name in CANDIDATES:
    wf = walk_forward_summary(name)
    fin = scores[name].get("final")
    row: Dict[str, Any] = {
        **wf,
        "val_thresholds": fin["thresholds"] if fin else None,
        "wf_folds": {
            f: {"score": scores[name][f]["score"], "exp": scores[name][f]["exp"], "thresholds": scores[name][f]["thresholds"]}
            for f in WF_FOLDS
            if f in scores[name]
        },
    }
    if name != ENSEMBLE_NAME and "up" in runs["final"].get(name, {}):
        r = runs["final"][name]
        up_c = corr(as_1d(r["up"].predict(X_val)), R_val[:, 0])
        dn_c = corr(as_1d(r["dn"].predict(X_val)), R_val[:, 1])
        row["upside_corr"] = up_c
        row["downside_corr"] = dn_c
        row["reg_score"] = float(np.nanmean([up_c, dn_c]))
    leaderboard[name] = row
    print(
        f"{name:9s} WF mean={wf['wf_mean_score']:+.6f} min={wf['wf_min_score']:+.6f} "
        f"pos_share={wf['wf_positive_share']:.2f} folds={wf['wf_folds_scored']:.0f} "
        f"val_thr={row['val_thresholds']} reg={row.get('reg_score', float('nan')):.3f}"
    )


def _selection_key(name: str) -> Tuple[float, float]:
    wf = leaderboard[name]
    return (wf["wf_mean_score"], wf["wf_min_score"])


cls_winner_name = max(CANDIDATES, key=_selection_key)
reg_candidates = [n for n in CANDIDATES if "reg_score" in leaderboard[n]]
reg_winner_name = max(reg_candidates, key=lambda n: float(leaderboard[n]["reg_score"]))
cls_winner = {"name": cls_winner_name, **leaderboard[cls_winner_name]}
reg_winner = {"name": reg_winner_name, "reg_score": leaderboard[reg_winner_name]["reg_score"]}
print("best classifier (walk-forward):", cls_winner_name, "| best regressor family (val):", reg_winner_name)


wf1: fit=1675 calib=341 eval=2270
wf2: fit=3646 calib=689 eval=2270
wf3: fit=5617 calib=1037 eval=2271
final: fit=7305 calib=1571 eval=1475
Fitting ['LR', 'RF', 'XGB', 'LGBM', 'CatBoost'] x ['wf1', 'wf2', 'wf3', 'final'] in parallel...


  done LR        wf1  (2s)


  done LR        wf2  (4s)


  done RF        wf1  (11s)


  done XGB       wf1  (19s)


...

  done CatBoost  wf3  (119s)


  done RF        final  (131s)


  done CatBoost  final  (147s)
LR        WF mean=-0.003937 min=-0.007036 pos_share=0.00 folds=2 val_thr={'LONG': 0.3540667248080535, 'SHORT': 0.3509856665702854} reg=0.275


RF        WF mean=-0.000990 min=-0.001264 pos_share=0.00 folds=2 val_thr={'LONG': 0.3274997356032872, 'SHORT': 0.32896132538096023} reg=0.357
XGB       WF mean=-0.000216 min=-0.000407 pos_share=0.00 folds=2 val_thr={'LONG': 0.4723896920681, 'SHORT': 0.40004709362983704} reg=0.273
LGBM      WF mean=-0.000624 min=-0.002373 pos_share=0.50 folds=2 val_thr={'LONG': 0.48569728038704424, 'SHORT': 0.4472229079302754} reg=0.271
CatBoost  WF mean=-0.000852 min=-0.004193 pos_share=0.50 folds=2 val_thr={'LONG': 0.35894924164654163, 'SHORT': 0.37738017682474184} reg=0.301
ENSEMBLE  WF mean=-0.001279 min=-0.003781 pos_share=0.50 folds=2 val_thr={'LONG': 0.35646578522477224, 'SHORT': 0.37487836857534323} reg=nan
best classifier (walk-forward): XGB | best regressor family (val): RF


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:
# CELL: EVAL
def _unscale_reg(pred: np.ndarray, col: int) -> np.ndarray:
    return as_1d(pred) * float(reg_std[0, col]) + float(reg_mean[0, col])


rows_test = all_rows[sel_test]
final_scores = {name: scores[name]["final"] for name in CANDIDATES}
win = final_scores[cls_winner_name]
pred_cls = win["pred"]
cm = np.asarray(win["confusion"])
print("TEST winner clf=", cls_winner_name, "thresholds=", win["thresholds"], "reg=", reg_winner_name)
print(
    f"test candles eligible for a signal (TP distance >= {MIN_SIGNAL_TP_PCT:.4%}): "
    f"{float(signal_eligible(rows_test).mean()):.1%}"
)
print("test confusion rows=true cols=pred LONG/SHORT/NO_TRADE")
print(cm)
print("test signals", {k: v for k, v in win["exp"].items()})

# Cost sensitivity of the winner's test signals (same decisions, different cost).
cost_sensitivity = {}
for c in CONFIG["cost_sensitivity_pct"]:
    rep = signal_expectancy(pred_cls, rows_test, cost=float(c))
    cost_sensitivity[f"{float(c):.4f}"] = {
        "score": expectancy_rank_score(rep),
        "LONG_expectancy": rep["LONG_expectancy"],
        "SHORT_expectancy": rep["SHORT_expectancy"],
    }
    print(f"cost {float(c):.4%}: score={cost_sensitivity[f'{float(c):.4f}']['score']:+.6f}")

member_test: Dict[str, Dict[str, Any]] = {}
for name in CANDIDATES:
    s = final_scores[name]
    member_test[name] = {**s["cls"], "exp": s["exp"], "score": s["score"], "thresholds": s["thresholds"]}
    if name != ENSEMBLE_NAME and "up" in runs["final"][name]:
        r = runs["final"][name]
        member_test[name]["upside_corr"] = corr(_unscale_reg(r["up"].predict(X_test), 0), r_test[:, 0])
        member_test[name]["downside_corr"] = corr(_unscale_reg(r["dn"].predict(X_test), 1), r_test[:, 1])
    print(
        name,
        "test",
        {
            "score": round(s["score"], 6) if np.isfinite(s["score"]) else s["score"],
            "LONG": (int(s["exp"]["LONG_n"]), round(s["exp"]["LONG_expectancy"], 6) if np.isfinite(s["exp"]["LONG_expectancy"]) else None),
            "SHORT": (int(s["exp"]["SHORT_n"]), round(s["exp"]["SHORT_expectancy"], 6) if np.isfinite(s["exp"]["SHORT_expectancy"]) else None),
            "P_LONG": round(s["cls"]["precision_LONG"], 3) if np.isfinite(s["cls"]["precision_LONG"]) else None,
            "P_SHORT": round(s["cls"]["precision_SHORT"], 3) if np.isfinite(s["cls"]["precision_SHORT"]) else None,
        },
    )

best_up = runs["final"][reg_winner_name]["up"]
best_dn = runs["final"][reg_winner_name]["dn"]
up_hat = _unscale_reg(best_up.predict(X_test), 0)
dn_hat = _unscale_reg(best_dn.predict(X_test), 1)
naive_up_mae = float(np.mean(np.abs(r_test[:, 0] - float(reg_mean[0, 0]))))
naive_dn_mae = float(np.mean(np.abs(r_test[:, 1] - float(reg_mean[0, 1]))))
reg_eval = {
    "upside_corr": corr(up_hat, r_test[:, 0]),
    "downside_corr": corr(dn_hat, r_test[:, 1]),
    "upside_mae_skill": 1.0 - float(np.mean(np.abs(r_test[:, 0] - up_hat))) / naive_up_mae,
    "downside_mae_skill": 1.0 - float(np.mean(np.abs(r_test[:, 1] - dn_hat))) / naive_dn_mae,
}
print("regression test", reg_eval)

eval_report = {
    "n_test": int(len(y_test)),
    "best_classifier": cls_winner_name,
    "best_regressor_family": reg_winner_name,
    "decision_rule": {
        "type": "probability_threshold_abstain",
        "thresholds": win["thresholds"],
        "min_signals": int(CONFIG["min_signals"]),
        "min_signal_tp_pct": MIN_SIGNAL_TP_PCT,
        "test_eligible_share": float(signal_eligible(rows_test).mean()),
    },
    "val_leaderboard": leaderboard,
    "test_members": member_test,
    "confusion": cm.tolist(),
    "precision_LONG": win["cls"]["precision_LONG"],
    "precision_SHORT": win["cls"]["precision_SHORT"],
    "n_LONG": win["cls"]["n_LONG"],
    "n_SHORT": win["cls"]["n_SHORT"],
    "balanced_accuracy": win["cls"]["balanced_accuracy"],
    "signal_expectancy": win["exp"],
    "score": win["score"],
    "cost_sensitivity": cost_sensitivity,
    "regression": reg_eval,
    "note": (
        "Classifier chosen by mean walk-forward thresholded expectancy (both sides, "
        f"min {int(CONFIG['min_signals'])} signals per side). Thresholds for the test come from "
        "validation only. Test is scored once. Balanced accuracy is diagnostic only."
    ),
}
print(json.dumps({k: v for k, v in eval_report.items() if k not in ("val_leaderboard", "test_members")}, indent=2, default=float))


TEST winner clf= XGB thresholds= {'LONG': 0.4723896920681, 'SHORT': 0.40004709362983704} reg= RF
test candles eligible for a signal (TP distance >= 0.6400%): 99.8%
test confusion rows=true cols=pred LONG/SHORT/NO_TRADE
[[ 76  83 294]
 [ 72  61 207]
 [102 114 466]]
test signals {'LONG_n': 250.0, 'LONG_expectancy': -0.0015463368042533897, 'LONG_tstat': -1.194823630908696, 'LONG_profit_factor': 0.8457442497114034, 'LONG_total_pct': -0.38658420106334745, 'SHORT_n': 258.0, 'SHORT_expectancy': -0.0033560460880354788, 'SHORT_tstat': -2.842170496283194, 'SHORT_profit_factor': 0.6758164941204458, 'SHORT_total_pct': -0.8658598907131535}
cost 0.1000%: score=-0.001851
cost 0.1600%: score=-0.002451
cost 0.2000%: score=-0.002851
LR test {'score': -0.003399, 'LONG': (449, -0.003628), 'SHORT': (278, -0.00317), 'P_LONG': 0.232, 'P_SHORT': 0.227}


RF test {'score': -0.000642, 'LONG': (555, -0.000702), 'SHORT': (264, -0.000583), 'P_LONG': 0.294, 'P_SHORT': 0.295}
XGB test {'score': -0.002451, 'LONG': (250, -0.001546), 'SHORT': (258, -0.003356), 'P_LONG': 0.304, 'P_SHORT': 0.236}
LGBM test {'score': -0.000891, 'LONG': (357, 0.001077), 'SHORT': (266, -0.002858), 'P_LONG': 0.342, 'P_SHORT': 0.259}
CatBoost test {'score': -0.001282, 'LONG': (536, 0.000232), 'SHORT': (229, -0.002797), 'P_LONG': 0.312, 'P_SHORT': 0.24}
ENSEMBLE test {'score': -0.001734, 'LONG': (507, -0.000305), 'SHORT': (254, -0.003162), 'P_LONG': 0.302, 'P_SHORT': 0.24}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


regression test {'upside_corr': 0.2831255327619778, 'downside_corr': 0.49978452827121533, 'upside_mae_skill': 0.022180637521468927, 'downside_mae_skill': 0.0974870391653313}
{
  "n_test": 1475,
  "best_classifier": "XGB",
  "best_regressor_family": "RF",
  "decision_rule": {
    "type": "probability_threshold_abstain",
    "thresholds": {
      "LONG": 0.4723896920681,
      "SHORT": 0.40004709362983704
    },
    "min_signals": 100,
    "min_signal_tp_pct": 0.0064,
    "test_eligible_share": 0.9979661016949153
  },
  "confusion": [
    [
      76,
      83,
      294
    ],
    [
      72,
      61,
      207
    ],
    [
      102,
      114,
      466
    ]
  ],
  "precision_LONG": 0.304,
  "precision_SHORT": 0.2364341085271318,
  "n_LONG": 250.0,
  "n_SHORT": 258.0,
  "balanced_accuracy": 0.3434888805366456,
  "signal_expectancy": {
    "LONG_n": 250.0,
    "LONG_expectancy": -0.0015463368042533897,
    "LONG_tstat": -1.194823630908696,
    "LONG_profit_factor": 0.8457442497114034,

In [ ]:
# CELL: PROMOTE
def promotion_gate(eval_rep: Dict[str, Any], lb_row: Dict[str, Any]) -> Dict[str, Any]:
    """Decide ``ready_to_promote`` from held-out evidence only.

    Requirements: positive thresholded expectancy on the single held-out test,
    every active side has >= ``min_signals`` trades and t-stat >= ``promote_min_tstat``,
    expectancy stays positive at the highest cost in ``cost_sensitivity_pct``, and the
    walk-forward positive share is >= ``promote_min_wf_positive_share``.
    """
    reasons: List[str] = []
    min_n = int(CONFIG["min_signals"])
    min_t = float(CONFIG["promote_min_tstat"])
    exp = eval_rep["signal_expectancy"]
    active_sides = [s for s in ("LONG", "SHORT") if float(exp.get(f"{s}_n", 0.0)) > 0]
    if not active_sides:
        reasons.append("no_signals_on_test")
    if not (np.isfinite(eval_rep["score"]) and eval_rep["score"] > 0.0):
        reasons.append("test_expectancy_not_positive")
    for side in active_sides:
        if float(exp[f"{side}_n"]) < min_n:
            reasons.append(f"{side.lower()}_too_few_test_signals")
        t = float(exp.get(f"{side}_tstat", float("nan")))
        if not (np.isfinite(t) and t >= min_t):
            reasons.append(f"{side.lower()}_tstat_below_{min_t:g}")
        if not (np.isfinite(exp[f"{side}_expectancy"]) and exp[f"{side}_expectancy"] > 0.0):
            reasons.append(f"{side.lower()}_expectancy_not_positive")
    worst_cost = max(float(c) for c in CONFIG["cost_sensitivity_pct"])
    worst = eval_rep["cost_sensitivity"][f"{worst_cost:.4f}"]["score"]
    if not (np.isfinite(worst) and worst > 0.0):
        reasons.append("fragile_to_higher_cost")
    if float(lb_row.get("wf_positive_share", 0.0)) < float(CONFIG["promote_min_wf_positive_share"]):
        reasons.append("walk_forward_not_consistently_positive")
    if float(lb_row.get("wf_folds_scored", 0.0)) < 2:
        reasons.append("walk_forward_insufficient_folds")
    return {
        "ready_to_promote": len(reasons) == 0,
        "reasons": reasons,
        "test_score": eval_rep["score"],
        "test_signals": {s: exp.get(f"{s}_n") for s in ("LONG", "SHORT")},
        "test_tstats": {s: exp.get(f"{s}_tstat") for s in ("LONG", "SHORT")},
        "wf_positive_share": lb_row.get("wf_positive_share"),
        "wf_mean_score": lb_row.get("wf_mean_score"),
        "worst_cost_score": worst,
    }


promotion = promotion_gate(eval_report, leaderboard[cls_winner_name])
ready_to_promote = bool(promotion["ready_to_promote"])
print(json.dumps(promotion, indent=2, default=float))
if ready_to_promote:
    print("VERDICT: held-out evidence supports paper-trading this bundle (still research-only here).")
else:
    print("VERDICT: NOT ready. Do not trade this bundle live. Reasons:", promotion["reasons"])


{
  "ready_to_promote": false,
  "reasons": [
    "test_expectancy_not_positive",
    "long_tstat_below_2",
    "long_expectancy_not_positive",
    "short_tstat_below_2",
    "short_expectancy_not_positive",
    "fragile_to_higher_cost",
    "walk_forward_not_consistently_positive"
  ],
  "test_score": -0.002451191446144434,
  "test_signals": {
    "LONG": 250.0,
    "SHORT": 258.0
  },
  "test_tstats": {
    "LONG": -1.194823630908696,
    "SHORT": -2.842170496283194
  },
  "wf_positive_share": 0.0,
  "wf_mean_score": -0.00021620385905574523,
  "worst_cost_score": -0.0028511914461444346
}
VERDICT: NOT ready. Do not trade this bundle live. Reasons: ['test_expectancy_not_positive', 'long_tstat_below_2', 'long_expectancy_not_positive', 'short_tstat_below_2', 'short_expectancy_not_positive', 'fragile_to_higher_cost', 'walk_forward_not_consistently_positive']


In [ ]:
# CELL: EXPORT
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
final_run = runs["final"][cls_winner_name]
if cls_winner_name == ENSEMBLE_NAME:
    clf_payload: Any = {
        "members": {m: runs["final"][m]["clf"] for m in final_run["members"]},
        "member_scalers": {m: runs["final"][m]["scaler"] for m in final_run["members"]},
    }
else:
    clf_payload = {"members": {cls_winner_name: final_run["clf"]}, "member_scalers": {cls_winner_name: final_run["scaler"]}}
bundle = {
    "best_classifier": cls_winner_name,
    "best_regressor_family": reg_winner_name,
    "clf": clf_payload,
    "thresholds": eval_report["decision_rule"]["thresholds"],
    "up": best_up,
    "dn": best_dn,
    "feat_mean": feat_mean,
    "feat_std": feat_std,
    "reg_mean": reg_mean,
    "reg_std": reg_std,
    "feature_cols": list(FEATURE_COLS),
    "label_names": list(LABEL_NAMES),
}
with (EXPORT_DIR / "best_models.pkl").open("wb") as handle:
    pickle.dump(bundle, handle)
(EXPORT_DIR / "leaderboard.json").write_text(
    json.dumps(leaderboard, indent=2, default=float), encoding="utf-8"
)
metadata = {
    "model_family": f"jacksparrow_opportunity_{CONFIG['resolution']}_tabular_ensemble",
    "ensemble": list(CONFIG["ensemble"]),
    "candidates": CANDIDATES,
    "best_classifier": cls_winner_name,
    "best_regressor_family": reg_winner_name,
    "timeframe": str(CONFIG["resolution"]),
    "resolution": str(CONFIG["resolution"]),
    "horizon_bars": int(CONFIG["horizon_bars"]),
    "horizon_minutes": int(CONFIG["horizon_minutes"]),
    "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
    "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
    "round_trip_cost_pct": float(CONFIG["round_trip_cost_pct"]),
    "feature_cols": list(FEATURE_COLS),
    "label_names": list(LABEL_NAMES),
    "decision_rule": {
        **eval_report["decision_rule"],
        "description": (
            f"At a closed {CONFIG['resolution']} candle: if tp_atr_mult * ATR(14) / close < min_signal_tp_pct emit "
            "NO_TRADE (quiet-market guard). Otherwise scale features with the member scaler(s), "
            "average member predict_proba when best_classifier == ENSEMBLE, then emit LONG when "
            "P(LONG) >= thresholds.LONG and P(LONG) > P(SHORT); SHORT symmetrically; else NO_TRADE. "
            "A null threshold disables that side."
        ),
    },
    "ready_to_promote": ready_to_promote,
    "promote_reason": (
        "held_out_gate_passed_paper_trade_first" if ready_to_promote else ";".join(promotion["reasons"])
    ),
    "promotion": promotion,
    "eval": eval_report,
    "label_gate": label_report,
    "walk_forward": {name: leaderboard[name]["wf_folds"] for name in CANDIDATES},
    "config": CONFIG,
}
(EXPORT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, default=float), encoding="utf-8"
)
(EXPORT_DIR / "eval_report.json").write_text(
    json.dumps(eval_report, indent=2, default=float), encoding="utf-8"
)
print("wrote", EXPORT_DIR / "best_models.pkl")
print("best classifier", cls_winner_name, "best regressor", reg_winner_name)
print("ready_to_promote=", ready_to_promote)
try:
    from google.colab import files
    import shutil

    shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=str(EXPORT_DIR))
    files.download(str(EXPORT_DIR) + ".zip")
except ImportError:
    print("Not Colab; files remain under", EXPORT_DIR)


wrote export/opportunity_2h_h48/best_models.pkl
best classifier XGB best regressor RF
ready_to_promote= False


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

*File Operation*: `download` on `/content/export/opportunity_2h_h48.zip`